# Edge-Native Hybrid ChakraModel Evaluation
This notebook evaluates the fully combined YOLO + ChakraTransformer architecture across **ALL 5 BENCHMARKS**.

In [ ]:
!pip install -q ultralytics timm opencv-python-headless torchvision gudhi

import sys
import os
import glob
import shutil
from pathlib import Path

print("Looking for attached code dataset...")
# Deep search for evaluate_all.py across all nested Kaggle folders
eval_script = glob.glob('/kaggle/input/**/src/evaluate_all.py', recursive=True)

if eval_script:
    # The parent of the 'src' folder is our root codebase directory
    src_dir = os.path.dirname(os.path.dirname(eval_script[0]))
    print(f"Found codebase at: {src_dir}")
    print("Copying to working directory to allow execution...")
    
    if os.path.exists('/kaggle/working/src'): shutil.rmtree('/kaggle/working/src')
    if os.path.exists('/kaggle/working/weights'): shutil.rmtree('/kaggle/working/weights')
    
    # Copy source and weights
    shutil.copytree(os.path.join(src_dir, 'src'), '/kaggle/working/src')
    shutil.copytree(os.path.join(src_dir, 'weights'), '/kaggle/working/weights')
    sys.path.insert(0, '/kaggle/working/src')
    print("✅ Setup Complete!")
else:
    print("ERROR: Could not find 'src/evaluate_all.py'.")
    print("Check the right sidebar -> 'Data' -> 'Datasets' to ensure you attached the Code Dataset!")


## 1. Setup Kaggle Datasets (Kvasir, ClinicDB, ETIS)


In [ ]:
os.makedirs('/kaggle/working/data', exist_ok=True)

def auto_symlink(dataset_name, internal_name):
    # Find any folder in /kaggle/input that contains dataset_name (case-insensitive)
    all_dirs = glob.glob('/kaggle/input/**/', recursive=True)
    ds_dirs = [d for d in all_dirs if dataset_name.lower() in d.lower()]
    
    target_dir = None
    for d in ds_dirs:
        subdirs = [os.path.basename(os.path.normpath(sub)).lower() for sub in glob.glob(d + '/*/')]
        if 'images' in subdirs or 'original' in subdirs or 'image' in subdirs:
            target_dir = d
            break
            
    if target_dir:
        target = f'/kaggle/working/data/{internal_name}'
        if not os.path.exists(target):
            os.symlink(target_dir, target)
            print(f'✅ Linked {internal_name} from {target_dir}')
    else:
        pass

auto_symlink('kvasir', 'kvasir-seg')
auto_symlink('clinic', 'cvc-clinicdb')
auto_symlink('etis', 'etis-larib')


## 2. Download Missing Benchmarks from Zenodo (ColonDB & 300)

In [ ]:
import urllib.request
import zipfile

print("Downloading missing CVC-ColonDB and CVC-300 from Zenodo...")
# Fixed Zenodo URL using 'records' instead of 'record'
url = "https://zenodo.org/records/3761817/files/TestDataset.zip"
zip_path = "/kaggle/working/TestDataset.zip"

if not os.path.exists(zip_path):
    try:
        urllib.request.urlretrieve(url, zip_path)
    except Exception as e:
        print(f"❌ Download failed: {e}")

if os.path.exists(zip_path):
    print("Extracting benchmarks...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/kaggle/working/raw_zenodo')
    
    for ds in ['CVC-ColonDB', 'CVC-300']:
        internal_name = ds.lower()
        tgt_dir = f"/kaggle/working/data/{internal_name}"
        os.makedirs(f"{tgt_dir}/images", exist_ok=True)
        os.makedirs(f"{tgt_dir}/masks", exist_ok=True)
        
        src_images = f"/kaggle/working/raw_zenodo/TestDataset/{ds}/images"
        src_masks = f"/kaggle/working/raw_zenodo/TestDataset/{ds}/masks"
        
        if os.path.exists(src_images):
            for f in os.listdir(src_images):
                shutil.copy2(os.path.join(src_images, f), os.path.join(tgt_dir, "images", f))
            for f in os.listdir(src_masks):
                shutil.copy2(os.path.join(src_masks, f), os.path.join(tgt_dir, "masks", f))
                
    print("✅ CVC-ColonDB and CVC-300 are ready!")


## 3. Run Final 5-Dataset Evaluation


In [ ]:
!python /kaggle/working/src/evaluate_all.py